# CD1 — Aula 06 · Laboratório: Validação cruzada

Nesta aula você pratica a **mecânica** da validação cruzada: k-fold, `cross_val_score` (quem separa treino/teste), média ± desvio, estratificação e o limite da classe rara, leave-one-out, k-fold repetido, o problema do grupo (GroupKFold), o tempo (TimeSeriesSplit) e a CV com Pipeline (sem vazar).

> A maldição do vencedor e a Nested CV ficam para a **Aula 07**.

Complete as células marcadas com `# TODO`. Rode tudo do começo ao fim.

In [49]:
import numpy as np, pandas as pd
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    StratifiedKFold, LeaveOneOut, RepeatedStratifiedKFold, GroupKFold, TimeSeriesSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
np.random.seed(0)

## Dados (fornecido)

Lemos a base **`dados_aula06.csv`** (desbalanceada, ~10% de positivos) e **separamos um teste final** que NÃO entra na validação cruzada.

In [50]:
df = pd.read_csv('dados_aula06.csv')
X = df.drop(columns='target').values
y = df['target'].values
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=0)
print('dev:', X_dev.shape, '| test:', X_test.shape, '| positivos em dev:', int(y_dev.sum()))

dev: (320, 8) | test: (80, 8) | positivos em dev: 34


## Ex 1 — Um corte só engana

Treine uma `LogisticRegression` com **5 sementes** diferentes de `train_test_split` (sobre `X_dev, y_dev`) e imprima a acurácia de cada. Repare como o número **balança** só por causa do sorteio.

In [51]:
accuracies = []
for i in range(5):
    X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=i)
    model = LogisticRegression(random_state=0, solver='liblinear')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    accuracies.append(accuracy)
    print(f'Acurácia com seed {i}: {accuracy:.4f}')

print(f'\nMédia das acurácias: {np.mean(accuracies):.4f}')
print(f'Desvio padrão das acurácias: {np.std(accuracies):.4f}')

Acurácia com seed 0: 0.8906
Acurácia com seed 1: 0.8750
Acurácia com seed 2: 0.8906
Acurácia com seed 3: 0.8750
Acurácia com seed 4: 0.9219

Média das acurácias: 0.8906
Desvio padrão das acurácias: 0.0171


## Ex 2 — k-fold com `cross_val_score`

Rode um **5-fold** e imprima as 5 notas, a **média** e o **desvio-padrão**. (Note: você não separa treino/teste à mão — o `cv=5` faz o rodízio sozinho.)

In [32]:
model = LogisticRegression(random_state=0, solver='liblinear')
scores = cross_val_score(model, X_dev, y_dev, cv=5, scoring='accuracy')
print(f'Acurácias das 5 dobras: {scores}')
print(f'Média: {scores.mean():.4f}')
print(f'Desvio-padrão: {scores.std():.4f}')

Acurácias das 5 dobras: [0.90625  0.90625  0.890625 0.875    0.890625]
Média: 0.8938
Desvio-padrão: 0.0117


## Ex 3 — Escolhendo o k

Compare **k = 5** e **k = 10**: imprima média ± desvio e lembre que k=10 faz mais treinos.

In [53]:
model = LogisticRegression(random_state=0, solver='liblinear')

# K=5
scores_5_fold = cross_val_score(model, X_dev, y_dev, cv=5, scoring='accuracy')
print(f'K=5: Média = {scores_5_fold.mean():.4f} \u00B1 {scores_5_fold.std():.4f}')

# K=10
scores_10_fold = cross_val_score(model, X_dev, y_dev, cv=10, scoring='accuracy')
print(f'K=10: Média = {scores_10_fold.mean():.4f} \u00B1 {scores_10_fold.std():.4f}')

K=5: Média = 0.8938 ± 0.0117
K=10: Média = 0.8938 ± 0.0286


## Ex 4 — StratifiedKFold preserva as classes

Conte quantos **positivos** caem em cada dobra de teste usando `KFold` (aleatório) e `StratifiedKFold`. Veja o estratificado equilibrar.

In [60]:
print('Contagem de positivos por dobra (KFold):')
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(kf.split(X_dev, y_dev)):
    y_test_fold = y_dev[test_index]
    print(f'  Dobra {fold+1}: {y_test_fold.sum()} positivos')

print('\nContagem de positivos por dobra (StratifiedKFold):')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(skf.split(X_dev, y_dev)):
    y_test_fold = y_dev[test_index]
    print(f'  Dobra {fold+1}: {y_test_fold.sum()} positivos')

Contagem de positivos por dobra (KFold):
  Dobra 1: 10 positivos
  Dobra 2: 12 positivos
  Dobra 3: 4 positivos
  Dobra 4: 4 positivos
  Dobra 5: 4 positivos

Contagem de positivos por dobra (StratifiedKFold):
  Dobra 1: 6 positivos
  Dobra 2: 7 positivos
  Dobra 3: 7 positivos
  Dobra 4: 7 positivos
  Dobra 5: 7 positivos


## Ex 5 — O limite da classe rara

Monte um conjunto com **apenas 4 positivos** e conte os positivos por dobra com `StratifiedKFold` para k = 3, 5 e 10. Confirme a regra **k ≤ nº de positivos**: a partir de k = 5 aparecem dobras com **0 positivos** (o sklearn ainda avisa `least populated class...`).

In [68]:
positive_indices = np.where(y_dev == 1)[0]

np.random.seed(0)
negative_indices = np.random.choice(np.where(y_dev == 0)[0], size=len(positive_indices), replace=False)


idx_pos = np.where(y_dev == 1)[0][:4]
idx_neg = np.where(y_dev == 0)[0][:4]

selected_indices = np.concatenate([idx_pos, idx_neg])
np.random.shuffle(selected_indices)

X_rare = X_dev[selected_indices]
y_rare = y_dev[selected_indices]

print(f'Total de positivos no conjunto raro: {y_rare.sum()}')

for k in [3, 5, 10]:
    print(f'\nStratifiedKFold com k = {k}:')
    try:
        skf_rare = StratifiedKFold(n_splits=k, shuffle=True, random_state=0)
        for fold, (train_index, test_index) in enumerate(skf_rare.split(X_rare, y_rare)):
            y_test_fold = y_rare[test_index]
            print(f'  Dobra {fold+1}: {y_test_fold.sum()} positivos')
    except ValueError as e:
        print(f'  Erro: {e}')
        print(f'  Isso demonstra a regra: n_splits={k} é maior que o número de positivos ({y_rare.sum()}). StratifiedKFold não consegue garantir que cada dobra de teste tenha pelo menos um exemplo da classe minoritária.')

Total de positivos no conjunto raro: 4

StratifiedKFold com k = 3:
  Dobra 1: 2 positivos
  Dobra 2: 1 positivos
  Dobra 3: 1 positivos

StratifiedKFold com k = 5:
  Erro: n_splits=5 cannot be greater than the number of members in each class.
  Isso demonstra a regra: n_splits=5 é maior que o número de positivos (4). StratifiedKFold não consegue garantir que cada dobra de teste tenha pelo menos um exemplo da classe minoritária.

StratifiedKFold com k = 10:
  Erro: Cannot have number of splits n_splits=10 greater than the number of samples: n_samples=8.
  Isso demonstra a regra: n_splits=10 é maior que o número de positivos (4). StratifiedKFold não consegue garantir que cada dobra de teste tenha pelo menos um exemplo da classe minoritária.


## Ex 6 — Leave-One-Out

Rode o **LOO** num subconjunto de 40 exemplos. Mostre que o nº de rodadas é n, que cada nota é **0 ou 1**, e a acurácia média.

In [69]:
n_subset = 40
X_loo = X_dev[:n_subset]
y_loo = y_dev[:n_subset]

loo = LeaveOneOut()
model = LogisticRegression(random_state=0, solver='liblinear')

scores_loo = cross_val_score(model, X_loo, y_loo, cv=loo, scoring='accuracy')

print(f'Número de rodadas (amostras em X_loo): {len(scores_loo)}')
print(f'Primeiras 10 notas (0 ou 1): {scores_loo[:10]}')
print(f'Acurácia média: {scores_loo.mean():.4f}')

Número de rodadas (amostras em X_loo): 40
Primeiras 10 notas (0 ou 1): [1. 0. 1. 1. 1. 1. 1. 1. 1. 1.]
Acurácia média: 0.9000


## Ex 7 — RepeatedKFold

Use `RepeatedStratifiedKFold` (5 dobras × 10 repetições = 50 notas). Imprima a média e o desvio — mais estável que uma repetição só.

In [71]:
model = LogisticRegression(random_state=0, solver='liblinear')
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
scores_rskf = cross_val_score(model, X_dev, y_dev, cv=rskf, scoring='accuracy')

print(f'Média das acurácias (50 rodadas): {scores_rskf.mean():.4f}')
print(f'Desvio-padrão das acurácias (50 rodadas): {scores_rskf.std():.4f}')

Média das acurácias (50 rodadas): 0.8944
Desvio-padrão das acurácias (50 rodadas): 0.0131


## Ex 8 — O problema do grupo e o GroupKFold

Crie `groups` (blocos de 5 linhas = um 'paciente'). Mostre que o `KFold` aleatório coloca o **mesmo grupo** nos dois lados (vaza) e o `GroupKFold` **não**.

In [73]:
n_samples = len(X_dev)
groups = np.repeat(np.arange(n_samples // 5), 5)[:n_samples]

print('KFold (vazamento):')
kf = KFold(n_splits=5, shuffle=True, random_state=0)
for fold, (train_index, test_index) in enumerate(kf.split(X_dev, y_dev, groups)):
    train_groups = set(groups[train_index])
    test_groups = set(groups[test_index])
    if len(train_groups.intersection(test_groups)) > 0:
        print(f'  Dobra {fold+1}: Grupos vazando: {train_groups.intersection(test_groups)}')
    else:
        print(f'  Dobra {fold+1}: Sem vazamento de grupos (inesperado com KFold)')

print('\nGroupKFold (sem vazamento):')
gkf = GroupKFold(n_splits=5)
for fold, (train_index, test_index) in enumerate(gkf.split(X_dev, y_dev, groups)):
    train_groups = set(groups[train_index])
    test_groups = set(groups[test_index])
    if len(train_groups.intersection(test_groups)) > 0:
        print(f'  Dobra {fold+1}: Grupos vazando: {train_groups.intersection(test_groups)} (ERRO - nao deveria vazar)')
    else:
        print(f'  Dobra {fold+1}: Sem vazamento de grupos.')

KFold (vazamento):
  Dobra 1: Grupos vazando: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(16), np.int64(18), np.int64(20), np.int64(21), np.int64(24), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(40), np.int64(41), np.int64(43), np.int64(45), np.int64(46), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(57), np.int64(61), np.int64(62), np.int64(63)}
  Dobra 2: Grupos vazando: {np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(14), np.int64(15), np.int64(17), np.int64(19), np.int64(20), np.int64(21), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(31), np.int64(32), np.int64(34), np.int64(35), 

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(


## Ex 9 — O tempo: TimeSeriesSplit

Mostre, dobra a dobra, que o treino é sempre o **passado** e o teste o **futuro** (o maior índice de treino é menor que o menor de teste).

In [74]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

print('TimeSeriesSplit:')
for fold, (train_index, test_index) in enumerate(tscv.split(X_dev)):
    print(f'  Dobra {fold+1}:')
    print(f'    Índice máx treino: {train_index.max()}')
    print(f'    Índice mín teste: {test_index.min()}')
    assert train_index.max() < test_index.min() # Confirma que treino é sempre antes do teste
    print(f'    Treino tem {len(train_index)} amostras, Teste tem {len(test_index)} amostras')

TimeSeriesSplit:
  Dobra 1:
    Índice máx treino: 54
    Índice mín teste: 55
    Treino tem 55 amostras, Teste tem 53 amostras
  Dobra 2:
    Índice máx treino: 107
    Índice mín teste: 108
    Treino tem 108 amostras, Teste tem 53 amostras
  Dobra 3:
    Índice máx treino: 160
    Índice mín teste: 161
    Treino tem 161 amostras, Teste tem 53 amostras
  Dobra 4:
    Índice máx treino: 213
    Índice mín teste: 214
    Treino tem 214 amostras, Teste tem 53 amostras
  Dobra 5:
    Índice máx treino: 266
    Índice mín teste: 267
    Treino tem 267 amostras, Teste tem 53 amostras


## Ex 10 — Por que o pré-processamento tem de ficar DENTRO da CV

Qualquer passo que **aprende dos dados** (selecionar atributos, padronizar, imputar) precisa ser refeito **em cada dobra, só com o treino**. Para ver o vazamento gritar, adicionamos **200 colunas de puro ruído** e usamos seleção de atributos. Compare selecionar **antes** da CV (olhando a base toda) com selecionar **dentro** do `Pipeline`.

In [75]:
from sklearn.feature_selection import SelectKBest, f_classif

# Adicionar 200 colunas de ruído
np.random.seed(0)
X_dev_noisy = np.hstack([X_dev, np.random.randn(X_dev.shape[0], 200)])

# Modelo e CV
model = LogisticRegression(random_state=0, solver='liblinear')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print('Cenário 1: Seleção de atributos ANTES da CV (vazamento de dados)')
# Selecionar os 5 melhores atributos em todo o X_dev_noisy
selector_pre_cv = SelectKBest(f_classif, k=5)
X_dev_selected_pre_cv = selector_pre_cv.fit_transform(X_dev_noisy, y_dev)

scores_pre_cv = cross_val_score(model, X_dev_selected_pre_cv, y_dev, cv=skf, scoring='accuracy')
print(f'  Média (pré-CV): {scores_pre_cv.mean():.4f} \u00B1 {scores_pre_cv.std():.4f}')

print('\nCenário 2: Seleção de atributos DENTRO do Pipeline (sem vazamento)')
# Pipeline que inclui a seleção de atributos e o modelo
pipeline_cv = Pipeline([
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', LogisticRegression(random_state=0, solver='liblinear'))
])

scores_in_cv = cross_val_score(pipeline_cv, X_dev_noisy, y_dev, cv=skf, scoring='accuracy')
print(f'  Média (dentro do Pipeline): {scores_in_cv.mean():.4f} \u00B1 {scores_in_cv.std():.4f}')

print('\nObserve que a acurácia no cenário 1 é artificialmente alta devido ao vazamento.')

Cenário 1: Seleção de atributos ANTES da CV (vazamento de dados)
  Média (pré-CV): 0.8906 ± 0.0140

Cenário 2: Seleção de atributos DENTRO do Pipeline (sem vazamento)
  Média (dentro do Pipeline): 0.8812 ± 0.0125

Observe que a acurácia no cenário 1 é artificialmente alta devido ao vazamento.


## Ex 11 — A nota final, no teste guardado

Monte o `Pipeline` (scaler + modelo), treine em todo o `X_dev` e meça **uma única vez** no `X_test`. Esse é o número honesto que você reporta.

In [76]:
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=0, solver='liblinear'))
])

# Treinar o pipeline em todo o conjunto de desenvolvimento (X_dev, y_dev)
pipeline_final.fit(X_dev, y_dev)

# Fazer previsões no conjunto de teste final (X_test)
y_pred_test = pipeline_final.predict(X_test)

# Calcular a acurácia no conjunto de teste
final_accuracy = accuracy_score(y_test, y_pred_test)

print(f'Acurácia final no conjunto de teste: {final_accuracy:.4f}')

Acurácia final no conjunto de teste: 0.8875


---
**Fecho.** Interno da aula: um corte só tem variância; a CV troca isso pela média de vários. Escolha o esquema pelo dado (estratificado em classificação, GroupKFold com repetição por entidade, TimeSeriesSplit com data) e rode sempre dentro de um Pipeline.